# Test Run

Make sure the RAG documents are inside the directory `./data` and the test question set `test_dataset.csv` is in the current working directory.

In [1]:
!pip install -q ollama
!pip install -q llama-index llama-index-readers-web llama-index-readers-file llama-index-llms-ollama llama-index-embeddings-huggingface
!pip list | grep -E 'llama-index|ollama'

llama-index                              0.14.20
llama-index-core                         0.14.20
llama-index-embeddings-huggingface       0.7.0
llama-index-embeddings-openai            0.6.0
llama-index-instrumentation              0.5.0
llama-index-llms-ollama                  0.10.1
llama-index-llms-openai                  0.7.5
llama-index-readers-file                 0.6.0
llama-index-readers-web                  0.6.0
llama-index-workflows                    2.17.3
ollama                                   0.6.1


In [2]:
import ollama
import pandas as pd
import itertools
from typing import List, Optional
import tracemalloc
from collections import defaultdict
from transformers import AutoTokenizer
from pydantic import Field

from llama_index.core.postprocessor.types import BaseNodePostprocessor
from llama_index.core.schema import NodeWithScore, QueryBundle
from llama_index.readers.web import SimpleWebPageReader
from llama_index.core.node_parser import SentenceSplitter, TokenTextSplitter, SemanticSplitterNodeParser, SentenceWindowNodeParser
from llama_index.core.utils import get_tokenizer
from llama_index.llms.ollama import Ollama
from llama_index.llms.openai import OpenAI
from llama_index.core import Document, SummaryIndex, Settings, PromptTemplate, SimpleDirectoryReader, VectorStoreIndex, set_global_tokenizer
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.ingestion import IngestionPipeline

INFO:datasets:TensorFlow version 2.19.0 available.
INFO:datasets:JAX version 0.7.2 available.


## Ollama

In [3]:
!apt-get update -qq && apt-get install -y lshw pciutils zstd -qq

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [4]:
!command -v ollama >/dev/null 2>&1 || curl -fsSL https://ollama.com/install.sh | sh

In [5]:
!nohup ollama serve > nohup.out 2>&1 &

In [6]:
!ollama pull mistral:7b
!ollama pull llama3.2:3b

In [7]:
response = ollama.chat(model='mistral:7b', messages=[
  {'role': 'user', 'content': 'The capital of France is'}
])

print('mistral:', response['message']['content'])

response = ollama.chat(model='llama3.2:3b', messages=[
  {'role': 'user', 'content': 'The capital of France is'}
])

print('llama3:', response['message']['content'])

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


mistral:  Paris, but it's important to note that the seat of the government is in a city within Paris, specifically known as Paris (or Seine-Saint-Denis department), while the city itself is divided into 20 administrative districts. The political capital of France is different from its cultural and commercial capital, which can sometimes be confused due to the similarity in names.


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


llama3: Paris.


## RAG

### LLMs, Embedings and Tokenizer

In [8]:
llms = {
    "mistral": Ollama(model="mistral:7b", request_timeout=120),
    "llama": Ollama(model="llama3.2:3b", request_timeout=120)
}

In [9]:
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.embed_model = embed_model

INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/model

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BA

In [10]:
model_name = "BAAI/bge-small-en-v1.5"
bge_tokenizer = AutoTokenizer.from_pretrained(model_name)
set_global_tokenizer(
    lambda text: bge_tokenizer.encode(text, add_special_tokens=False)
)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


### Chunking Strategies

In [11]:
chunk_strategies = {
    "SentenceSplitter(chunk_size=128, chunk_overlap=10)": SentenceSplitter(chunk_size=128, chunk_overlap=10),
    "SentenceSplitter(chunk_size=256, chunk_overlap=10)": SentenceSplitter(chunk_size=256, chunk_overlap=10),
    "SentenceSplitter(chunk_size=512, chunk_overlap=10)": SentenceSplitter(chunk_size=512, chunk_overlap=10),
    "SemanticSplitterNodeParser": SemanticSplitterNodeParser(embed_model=embed_model),
    "SentenceWindowNodeParser": SentenceWindowNodeParser()
}

### Indexing

In [12]:
def create_index(documents, spitter):
    pipeline = IngestionPipeline(
        transformations=[spitter, embed_model]
    )
    nodes = pipeline.run(documents=documents)
    index = VectorStoreIndex(nodes)

    return index

### Token Budget

In [13]:
class TokenBudgetPostprocessor(BaseNodePostprocessor):
    token_budget: int = 1024
    stats_total_tokens: list = Field(default_factory=list)
    stats_total_chunks: list = Field(default_factory=list)

    def _postprocess_nodes(self, nodes, query_bundle):
        tokenizer = get_tokenizer()
        total_tokens = 0
        filtered = []

        # must be sorted in the descending order of the similiarity score
        nodes = sorted(nodes, key=lambda n: n.score or 0.0, reverse=True)

        # filter the nodes by the token budget
        for node in nodes:
            content = node.node.get_content(metadata_mode="none")
            token_count = len(tokenizer(content))

            if total_tokens + token_count > self.token_budget:
                break

            total_tokens += token_count
            filtered.append(node)

        self.stats_total_tokens.append(total_tokens)
        self.stats_total_chunks.append(len(filtered))

        return filtered

### Answer Generation

In [14]:
def profile_memory(run):
    tracemalloc.start()
    snapshot_before = tracemalloc.take_snapshot()

    result = run()

    snapshot_after = tracemalloc.take_snapshot()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    return result, peak

In [15]:
df_testcases = pd.read_csv('./test_dataset.csv')
df_testcases.sample(5)

,document,question_type,question,reference_answer
1,Wiki,single_hop_factual,How many full-time students did the polytechni...,480 full-time students.
11,Curriculum,single_hop_factual,What is the maximum period of study allowed fo...,8 years
18,Curriculum,multi_task,Please answer the following three questions:\n...,1. The maximum period of study is 8 years.\n2....
23,Magazine,summarization,What are the key benefits and challenges of op...,The key benefits include making innovations ac...
26,Magazine,needle_in_haystack,What is the name of the college at the Univers...,"Lucy Cavendish College, whose president Dame M..."


In [16]:
configs = {
    "llms": llms.keys(),
    "chunk_strategies": chunk_strategies.keys()
}
total_configs = list(itertools.product(*configs.values()))
documents = SimpleDirectoryReader(
    "./data",
    required_exts=[".md"],
    filename_as_id=True
).load_data()
df_answers = df_testcases.copy()

for i, (conf_llm, conf_chunk_strat) in enumerate(total_configs):
    print(f'generating answers on config {i}/{len(total_configs)}')
    chunk_strat = chunk_strategies[conf_chunk_strat]
    index = create_index(documents, chunk_strat)
    llm = llms[conf_llm]

    budget_pp = TokenBudgetPostprocessor(token_budget=1500)
    query_engine = index.as_query_engine(
        llm=llm,
        similarity_top_k=50,
        node_postprocessors=[budget_pp]
    )

    answers = []
    contexts = []
    mem_usages = []

    for row in df_testcases.itertuples():
        question = row.question
        response, mem_usage = profile_memory(lambda: query_engine.query(question))

        answers.append(str(response))
        contexts.append([node.get_content() for node in response.source_nodes])
        mem_usages.append(mem_usage)

    df_answers[f'{conf_llm}:{conf_chunk_strat}:answer'] = answers
    df_answers[f'{conf_llm}:{conf_chunk_strat}:contexts'] = contexts
    df_answers[f'{conf_llm}:{conf_chunk_strat}:total_token'] = budget_pp.stats_total_tokens
    df_answers[f'{conf_llm}:{conf_chunk_strat}:num_chunk'] = budget_pp.stats_total_chunks
    df_answers[f'{conf_llm}:{conf_chunk_strat}:mem_usage'] = mem_usages

print('answer generation complete')

Token indices sequence length is longer than the specified maximum sequence length for this model (2121 > 512). Running this sequence through the model will result in indexing errors


generating answers on config 0/10


INFO:httpx:HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://loc

generating answers on config 1/10


INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://loc

...

generating answers on config 8/10


INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://loc

generating answers on config 9/10


INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://loc

answer generation complete


In [17]:
df_answers.to_csv('./generated_answer.csv', index=False)
df_answers.to_json('./generated_answer.json', orient='records')